In [1]:
import os
import json
import numpy as np
import pandas as pd

DATA_DIR = "/kaggle/input/competitions/fds-pokemon-battles-prediction-2026"

print("Files:")
print(os.listdir(DATA_DIR))

train_path = os.path.join(DATA_DIR, "train.jsonl")
test_path = os.path.join(DATA_DIR, "test.jsonl")

def load_jsonl(path):
    with open(path, "r") as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = load_jsonl(train_path)
test_data = load_jsonl(test_path)

print("\nTrain battles:", len(train_data))
print("Test battles:", len(test_data))

print("\nTrain keys:")
print(train_data[0].keys())

print("\nExample battle:")
print(json.dumps(train_data[0], indent=2)[:3000])

Files:
['sample_submission.csv', 'test.jsonl', 'train.jsonl', 'type_chart_gen2.csv']

Train battles: 10000
Test battles: 5000

Train keys:
dict_keys(['battle_id', 'p1_team_details', 'p2_lead_details', 'battle_timeline', 'player_won'])

Example battle:
{
  "battle_id": 0,
  "p1_team_details": [
    {
      "name": "tyranitar",
      "level": 100,
      "types": [
        "dark",
        "rock"
      ],
      "base_hp": 100,
      "base_atk": 134,
      "base_def": 110,
      "base_spa": 95,
      "base_spd": 100,
      "base_spe": 61,
      "item": "leftovers"
    },
    {
      "name": "typhlosion",
      "level": 100,
      "types": [
        "fire",
        "notype"
      ],
      "base_hp": 78,
      "base_atk": 84,
      "base_def": 78,
      "base_spa": 109,
      "base_spd": 85,
      "base_spe": 100,
      "item": "mintberry"
    },
    {
      "name": "golem",
      "level": 100,
      "types": [
        "ground",
        "rock"
      ],
      "base_hp": 80,
      "base_atk": 1

In [2]:
# Inspect the structure across multiple battles

print("PLAYER 1 TEAM")
print("Team size:", len(train_data[0]["p1_team_details"]))
print("Pokémon fields:")
print(train_data[0]["p1_team_details"][0].keys())

print("\nPLAYER 2 LEAD")
print("Fields:")
print(train_data[0]["p2_lead_details"].keys())

print("\nBATTLE TIMELINE")
print("Number of turns:", len(train_data[0]["battle_timeline"]))
print("Timeline event fields:")
print(train_data[0]["battle_timeline"][0].keys())

print("\nPLAYER 1 POKÉMON STATE")
print(train_data[0]["battle_timeline"][0]["p1_pokemon_state"])

print("\nPLAYER 1 MOVE")
print(train_data[0]["battle_timeline"][0]["p1_move_details"])

print("\nPLAYER 2 POKÉMON STATE")
print(train_data[0]["battle_timeline"][0]["p2_pokemon_state"])

print("\nPLAYER 2 MOVE")
print(train_data[0]["battle_timeline"][0]["p2_move_details"])


# Check consistency across the dataset
print("\nDATASET CONSISTENCY")

print("Train team sizes:",
      pd.Series([len(b["p1_team_details"]) for b in train_data]).value_counts().to_dict())

print("Train timeline lengths:")
print(pd.Series([len(b["battle_timeline"]) for b in train_data]).describe())

print("\nTest team sizes:",
      pd.Series([len(b["p1_team_details"]) for b in test_data]).value_counts().to_dict())

print("Test timeline lengths:")
print(pd.Series([len(b["battle_timeline"]) for b in test_data]).describe())


# Check target distribution
y = np.array([b["player_won"] for b in train_data])

print("\nTARGET DISTRIBUTION:")
print(pd.Series(y).value_counts().sort_index())

PLAYER 1 TEAM
Team size: 6
Pokémon fields:
dict_keys(['name', 'level', 'types', 'base_hp', 'base_atk', 'base_def', 'base_spa', 'base_spd', 'base_spe', 'item'])

PLAYER 2 LEAD
Fields:
dict_keys(['name', 'level', 'types', 'base_hp', 'base_atk', 'base_def', 'base_spa', 'base_spd', 'base_spe'])

BATTLE TIMELINE
Number of turns: 30
Timeline event fields:
dict_keys(['turn', 'p1_pokemon_state', 'p1_move_details', 'p2_pokemon_state', 'p2_move_details'])

PLAYER 1 POKÉMON STATE
{'name': 'tyranitar', 'hp_pct': 1.0, 'status': 'nostatus', 'effects': ['noeffect'], 'boosts': {'atk': 0, 'def': 0, 'spa': 0, 'spd': 0, 'spe': 0}}

PLAYER 1 MOVE
{'name': 'earthquake', 'type': 'ground', 'category': 'physical', 'base_power': 100, 'accuracy': 1.0, 'priority': 0}

PLAYER 2 POKÉMON STATE
{'name': 'forretress', 'hp_pct': 0.86, 'status': 'nostatus', 'effects': ['noeffect'], 'boosts': {'atk': 0, 'def': 0, 'spa': 0, 'spd': 0, 'spe': 0}}

PLAYER 2 MOVE
{'name': 'toxic', 'type': 'poison', 'category': 'status', 'bas

In [3]:
from collections import Counter
import numpy as np
import pandas as pd

STAT_FIELDS = [
    "base_hp", "base_atk", "base_def",
    "base_spa", "base_spd", "base_spe"
]

def add_counter_features(features, prefix, values):
    counts = Counter(str(v) for v in values if v is not None)
    for value, count in counts.items():
        key = f"{prefix}_{value}".lower().replace(" ", "_")
        features[key] = count


def extract_battle_features(battle):
    f = {}

    # 1. Player 1 team
    team = battle["p1_team_details"]
    f["team_size"] = len(team)

    for stat in STAT_FIELDS:
        values = np.array([
            float(p.get(stat, 0) or 0) for p in team
        ])
        f[f"team_{stat}_mean"] = values.mean()
        f[f"team_{stat}_min"] = values.min()
        f[f"team_{stat}_max"] = values.max()
        f[f"team_{stat}_sum"] = values.sum()
        f[f"team_{stat}_std"] = values.std()

    for p in team:
        for stat in STAT_FIELDS:
            key = f"team_best_{stat}"
            f[key] = max(
                f.get(key, 0),
                float(p.get(stat, 0) or 0)
            )

    add_counter_features(
        f, "team_type",
        [t for p in team for t in p.get("types", [])]
    )
    add_counter_features(
        f, "team_item",
        [p.get("item", "noitem") for p in team]
    )
    add_counter_features(
        f, "team_pokemon",
        [p.get("name", "unknown") for p in team]
    )

    # 2. Opponent lead
    lead = battle["p2_lead_details"]

    for stat in STAT_FIELDS:
        f[f"p2_lead_{stat}"] = float(lead.get(stat, 0) or 0)

    add_counter_features(f, "p2_lead_type", lead.get("types", []))
    f["p2_lead_level"] = float(lead.get("level", 0) or 0)

    for stat in STAT_FIELDS:
        f[f"team_minus_lead_{stat}"] = (
            f[f"team_{stat}_mean"] - f[f"p2_lead_{stat}"]
        )

    # 3. Battle timeline
    timeline = battle["battle_timeline"]
    n = len(timeline)

    for player in ["p1", "p2"]:

        states = [
            turn.get(f"{player}_pokemon_state") or {}
            for turn in timeline
        ]

        moves = [
            turn.get(f"{player}_move_details") or {}
            for turn in timeline
        ]

        hp = np.array([
            float(s.get("hp_pct", 0) or 0)
            for s in states
        ])

        # Existing HP features
        f[f"{player}_hp_first"] = hp[0]
        f[f"{player}_hp_last"] = hp[-1]
        f[f"{player}_hp_mean"] = hp.mean()
        f[f"{player}_hp_min"] = hp.min()
        f[f"{player}_hp_max"] = hp.max()
        f[f"{player}_hp_std"] = hp.std()
        f[f"{player}_hp_delta"] = hp[-1] - hp[0]
        f[f"{player}_hp_loss"] = hp[0] - hp[-1]

        diff = np.diff(hp)

        f[f"{player}_hp_increases"] = int(np.sum(diff > 0))
        f[f"{player}_hp_decreases"] = int(np.sum(diff < 0))
        f[f"{player}_hp_unchanged"] = int(np.sum(diff == 0))

        # NEW: HP trajectory features
        x = np.arange(n)

        if n > 1:
            f[f"{player}_hp_slope"] = np.polyfit(x, hp, 1)[0]
        else:
            f[f"{player}_hp_slope"] = 0

        f[f"{player}_hp_max_drop"] = max(0, -diff.min()) if len(diff) else 0
        f[f"{player}_hp_max_recovery"] = max(0, diff.max()) if len(diff) else 0

        f[f"{player}_hp_abs_change_mean"] = (
            np.abs(diff).mean() if len(diff) else 0
        )

        f[f"{player}_hp_volatility"] = (
            np.std(diff) if len(diff) else 0
        )

        f[f"{player}_hp_below_25"] = int(np.sum(hp < 0.25))
        f[f"{player}_hp_below_50"] = int(np.sum(hp < 0.50))
        f[f"{player}_hp_above_90"] = int(np.sum(hp > 0.90))

        f[f"{player}_hp_recovery_total"] = (
            np.maximum(diff, 0).sum()
        )

        f[f"{player}_hp_damage_total"] = (
            np.maximum(-diff, 0).sum()
        )

        # Early / middle / late HP
        sections = {
            "early": hp[:10],
            "middle": hp[10:20],
            "late": hp[20:30]
        }

        for label, section in sections.items():
            if len(section):
                f[f"{player}_hp_{label}_mean"] = section.mean()
                f[f"{player}_hp_{label}_min"] = section.min()
                f[f"{player}_hp_{label}_last"] = section[-1]
                f[f"{player}_hp_{label}_delta"] = (
                    section[-1] - section[0]
                )

        f[f"{player}_hp_momentum"] = (
            sections["late"].mean() - sections["early"].mean()
        )

        # Active Pokémon and switching
        active_names = [
            s.get("name", "unknown") for s in states
        ]

        f[f"{player}_unique_active"] = len(set(active_names))

        f[f"{player}_switches"] = sum(
            active_names[i] != active_names[i - 1]
            for i in range(1, len(active_names))
        )

        add_counter_features(f, f"{player}_active", active_names)

        # Status
        add_counter_features(
            f, f"{player}_status",
            [s.get("status", "unknown") for s in states]
        )

        # Effects
        add_counter_features(
            f, f"{player}_effect",
            [e for s in states for e in s.get("effects", [])]
        )

        # Boosts
        for stat in ["atk", "def", "spa", "spd", "spe"]:
            values = np.array([
                float((s.get("boosts") or {}).get(stat, 0) or 0)
                for s in states
            ])

            f[f"{player}_boost_{stat}_mean"] = values.mean()
            f[f"{player}_boost_{stat}_min"] = values.min()
            f[f"{player}_boost_{stat}_max"] = values.max()
            f[f"{player}_boost_{stat}_last"] = values[-1]

        # Moves
        powers = np.array([
            float(m.get("base_power", 0) or 0)
            for m in moves
        ])

        accuracies = np.array([
            float(m.get("accuracy", 0) or 0)
            for m in moves
        ])

        priorities = np.array([
            float(m.get("priority", 0) or 0)
            for m in moves
        ])

        f[f"{player}_move_power_mean"] = powers.mean()
        f[f"{player}_move_power_max"] = powers.max()
        f[f"{player}_move_power_sum"] = powers.sum()
        f[f"{player}_move_power_nonzero"] = int(np.sum(powers > 0))
        f[f"{player}_move_accuracy_mean"] = accuracies.mean()
        f[f"{player}_move_priority_mean"] = priorities.mean()

        add_counter_features(
            f, f"{player}_move_type",
            [m.get("type", "unknown") for m in moves]
        )

        add_counter_features(
            f, f"{player}_move_category",
            [m.get("category", "unknown") for m in moves]
        )

        add_counter_features(
            f, f"{player}_move_name",
            [m.get("name", "unknown") for m in moves]
        )

    # 4. Relative features
    f["hp_mean_difference"] = f["p1_hp_mean"] - f["p2_hp_mean"]
    f["hp_last_difference"] = f["p1_hp_last"] - f["p2_hp_last"]
    f["hp_min_difference"] = f["p1_hp_min"] - f["p2_hp_min"]
    f["hp_delta_difference"] = f["p1_hp_delta"] - f["p2_hp_delta"]
    f["switch_difference"] = f["p1_switches"] - f["p2_switches"]

    f["move_power_difference"] = (
        f["p1_move_power_mean"] - f["p2_move_power_mean"]
    )

    # NEW: Relative trajectory features
    for suffix in [
        "hp_slope",
        "hp_volatility",
        "hp_max_drop",
        "hp_recovery_total",
        "hp_damage_total",
        "hp_below_25",
        "hp_below_50",
        "hp_momentum"
    ]:
        f[f"{suffix}_difference"] = (
            f[f"p1_{suffix}"] - f[f"p2_{suffix}"]
        )

    return f


# 5. Extract features
print("Extracting training features...")

train_features = pd.DataFrame([
    extract_battle_features(b) for b in train_data
])

print("Extracting test features...")

test_features = pd.DataFrame([
    extract_battle_features(b) for b in test_data
])

# Align columns
all_columns = sorted(
    set(train_features.columns) | set(test_features.columns)
)

X = train_features.reindex(columns=all_columns, fill_value=0)
X_test = test_features.reindex(columns=all_columns, fill_value=0)

X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

y = np.array([b["player_won"] for b in train_data])

# Diagnostics
print("\nFEATURE EXTRACTION COMPLETE")
print("X:", X.shape)
print("X_test:", X_test.shape)
print("Target:", y.shape)

print("Variable features:", (X.nunique() > 1).sum())
print("Constant features:", (X.nunique() <= 1).sum())
print("Missing values:", X.isna().sum().sum())
print("Train/test match:", list(X.columns) == list(X_test.columns))

Extracting training features...
Extracting test features...

FEATURE EXTRACTION COMPLETE
X: (10000, 1591)
X_test: (5000, 1591)
Target: (10000,)
Variable features: 1532
Constant features: 59
Missing values: 0
Train/test match: True


In [4]:
diagnostics = pd.DataFrame({
    "unique": X.nunique(),
    "missing": X.isna().sum(),
    "std": X.std()
})

print("Total features:", X.shape[1])
print("Variable features:", (X.nunique() > 1).sum())
print("Constant features:", (X.nunique() <= 1).sum())
print("Missing values:", X.isna().sum().sum())
print("Train/test columns match:", X.columns.equals(X_test.columns))

print("\n20 least-variable features:")
print(diagnostics.sort_values("unique").head(20))

print("\n20 most-variable features:")
print(diagnostics.sort_values("unique", ascending=False).head(20))

Total features: 1591
Variable features: 1532
Constant features: 59
Missing values: 0
Train/test columns match: True

20 least-variable features:
                          unique  missing  std
team_size                      1        0  0.0
p1_active_celebi               1        0  0.0
p1_active_alakazammega         1        0  0.0
p1_active_cleffa               1        0  0.0
p2_move_name_spikecannon       1        0  0.0
p1_move_name_visegrip          1        0  0.0
p1_move_name_sludge            1        0  0.0
p1_move_name_stomp             1        0  0.0
p2_active_celebi               1        0  0.0
p2_active_charizardmegay       1        0  0.0
p2_active_cyndaquil            1        0  0.0
team_pokemon_horsea            1        0  0.0
p2_active_hooh                 1        0  0.0
team_pokemon_cleffa            1        0  0.0
p1_move_name_poisongas         1        0  0.0
p2_active_igglybuff            1        0  0.0
p2_active_kabuto               1        0  0.0
p2_active

In [5]:
import json
import glob
import os
import time
import warnings
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
t_start = time.time()
SEED = 42

# 1. Locate Competition Dataset Paths Automatically
train_paths = glob.glob('/kaggle/input/**/train.jsonl', recursive=True)
test_paths = glob.glob('/kaggle/input/**/test.jsonl', recursive=True)
TRAIN_PATH = train_paths[0]
TEST_PATH = test_paths[0]
BASE = os.path.dirname(TRAIN_PATH)

# Supplied Generation 2 Type Chart (100% Rule-Compliant)
GEN2_TYPE_CHART = {
    'normal': {'rock': 0.5, 'ghost': 0.0, 'steel': 0.5},
    'fire': {'fire': 0.5, 'water': 0.5, 'grass': 2.0, 'ice': 2.0, 'bug': 2.0, 'rock': 0.5, 'dragon': 0.5, 'steel': 2.0},
    'water': {'fire': 2.0, 'water': 0.5, 'grass': 0.5, 'ground': 2.0, 'rock': 2.0, 'dragon': 0.5},
    'electric': {'water': 2.0, 'electric': 0.5, 'grass': 0.5, 'ground': 0.0, 'flying': 2.0, 'dragon': 0.5},
    'grass': {'fire': 0.5, 'water': 2.0, 'grass': 0.5, 'poison': 0.5, 'ground': 2.0, 'flying': 0.5, 'bug': 0.5, 'rock': 2.0, 'dragon': 0.5, 'steel': 0.5},
    'ice': {'fire': 0.5, 'water': 0.5, 'grass': 2.0, 'ice': 0.5, 'ground': 2.0, 'flying': 2.0, 'dragon': 2.0, 'steel': 0.5},
    'fighting': {'normal': 2.0, 'ice': 2.0, 'poison': 0.5, 'flying': 0.5, 'psychic': 0.5, 'bug': 0.5, 'rock': 2.0, 'ghost': 0.0, 'dark': 2.0, 'steel': 2.0},
    'poison': {'grass': 2.0, 'poison': 0.5, 'ground': 0.5, 'rock': 0.5, 'ghost': 0.5, 'steel': 0.0},
    'ground': {'fire': 2.0, 'electric': 2.0, 'grass': 0.5, 'poison': 2.0, 'flying': 0.0, 'bug': 0.5, 'rock': 2.0, 'steel': 2.0},
    'flying': {'electric': 0.5, 'grass': 2.0, 'fighting': 2.0, 'bug': 2.0, 'rock': 0.5, 'steel': 0.5},
    'psychic': {'fighting': 2.0, 'poison': 2.0, 'psychic': 0.5, 'dark': 0.0, 'steel': 0.5},
    'bug': {'fire': 0.5, 'grass': 2.0, 'fighting': 0.5, 'poison': 0.5, 'flying': 0.5, 'psychic': 2.0, 'ghost': 0.5, 'dark': 2.0, 'steel': 0.5},
    'rock': {'fire': 2.0, 'ice': 2.0, 'fighting': 0.5, 'ground': 0.5, 'flying': 2.0, 'bug': 2.0, 'steel': 0.5},
    'ghost': {'normal': 0.0, 'psychic': 2.0, 'ghost': 2.0, 'dark': 0.5, 'steel': 0.5},
    'dragon': {'dragon': 2.0, 'steel': 0.5},
    'dark': {'fighting': 0.5, 'psychic': 2.0, 'ghost': 2.0, 'dark': 0.5, 'steel': 0.5},
    'steel': {'fire': 0.5, 'water': 0.5, 'electric': 0.5, 'ice': 2.0, 'rock': 2.0, 'steel': 0.5}
}

RECOVERY_MOVES = {'recover', 'rest', 'softboiled', 'milkdrink', 'moonlight', 'synthesis', 'morningsun', 'leechseed', 'pain_split', 'painsplit'}
PHAZING_HAZARD_MOVES = {'spikes', 'roar', 'whirlwind', 'rapidspin', 'rapid_spin', 'toxic', 'thunderwave', 'thunder_wave', 'stunspore', 'sleeppowder', 'spore', 'lovelykiss', 'hypnosis', 'encore', 'perishsong', 'destinybond', 'explosion', 'selfdestruct'}
SETUP_MOVES = {'curse', 'swordsdance', 'swords_dance', 'bellydrum', 'belly_drum', 'agility', 'growth', 'amnesia', 'barrier', 'acidarmor', 'substitute', 'batonpass', 'baton_pass'}
STAGE_MULT = {-6: 0.25, -5: 0.28, -4: 0.33, -3: 0.40, -2: 0.50, -1: 0.66, 0: 1.0, 1: 1.5, 2: 2.0, 3: 2.5, 4: 3.0, 5: 3.5, 6: 4.0}
STATUS_SEVERITY = {'nostatus': 0.0, 'par': 1.0, 'psn': 1.0, 'brn': 1.5, 'tox': 1.8, 'slp': 2.0, 'frz': 2.5, 'fnt': 3.0, 'fainted': 3.0}

def get_type_mult(atk_type, def_types):
    if not atk_type or not def_types:
        return 1.0
    at = str(atk_type).lower().strip()
    mult = 1.0
    for dt in def_types:
        if not dt or dt == 'notype': continue
        mult *= GEN2_TYPE_CHART.get(at, {}).get(str(dt).lower().strip(), 1.0)
    return mult

def best_dual_type_mult(atk_types, def_types):
    if not atk_types or not def_types:
        return 1.0
    return max([get_type_mult(at, def_types) for at in atk_types if at and at != 'notype'] or [1.0])

# 2. Build Internal Pokédex Strictly from train.jsonl
print("📚 Building Internal Pokédex strictly from train.jsonl...")
INTERNAL_POKEDEX = {}
with open(TRAIN_PATH, 'r') as f:
    for line in f:
        row = json.loads(line)
        for mon in (row.get('p1_team_details') or []) + [row.get('p2_lead_details') or {}]:
            if isinstance(mon, dict) and mon.get('name'):
                name = str(mon['name']).lower().strip()
                if name not in INTERNAL_POKEDEX:
                    hp = float(mon.get('base_hp', 80))
                    atk = float(mon.get('base_atk', 80))
                    df = float(mon.get('base_def', 80))
                    spa = float(mon.get('base_spa', 80))
                    spd = float(mon.get('base_spd', 80))
                    spe = float(mon.get('base_spe', 80))
                    types = [t.lower() for t in (mon.get('types') or []) if t and t.lower() != 'notype']
                    INTERNAL_POKEDEX[name] = {
                        'hp': hp, 'atk': atk, 'def': df, 'spa': spa, 'spd': spd, 'spe': spe,
                        'bst': hp + atk + df + spa + spd + spe,
                        'bulk': hp * (df + spd),
                        'off': max(atk, spa),
                        'types': types
                    }
DEFAULT_MON_INFO = {'hp': 80.0, 'atk': 80.0, 'def': 80.0, 'spa': 80.0, 'spd': 80.0, 'spe': 80.0, 'bst': 480.0, 'bulk': 12800.0, 'off': 80.0, 'types': []}
print(f"   ✅ Indexed {len(INTERNAL_POKEDEX)} species from train.jsonl!")

# 3. Extract Full V2 Metagame + V3 Checkpoint Trajectories + 20 Tactical Categoricals
print("🚀 Extracting V2 Metagame + V3 Checkpoint Trajectories + 20 Tactical Categoricals...")

def extract_v4_features(jsonl_path, is_train=True):
    rows_feats = []
    cat_rows = []
    token_records = []
    corpus_texts = []
    targets = []
    
    with open(jsonl_path, 'r') as f:
        for line in f:
            row = json.loads(line)
            if is_train and 'player_won' in row:
                targets.append(int(row['player_won']))
                
            d = {}
            p1_team = row.get('p1_team_details', []) or []
            p2_lead = row.get('p2_lead_details', {}) or {}
            timeline = row.get('battle_timeline', []) or []
            
            p1_names, p1_items = [], []
            p2_lead_name = str(p2_lead.get('name', 'unknown')).lower()
            p2_lead_types = [t.lower() for t in (p2_lead.get('types') or []) if t and t.lower() != 'notype']
            p2_hp = float(p2_lead.get('base_hp', 80))
            p2_atk = float(p2_lead.get('base_atk', 80))
            p2_def = float(p2_lead.get('base_def', 80))
            p2_spa = float(p2_lead.get('base_spa', 80))
            p2_spd = float(p2_lead.get('base_spd', 80))
            p2_spe = float(p2_lead.get('base_spe', 80))
            p2_bst = p2_hp + p2_atk + p2_def + p2_spa + p2_spd + p2_spe
            
            d['exact_p2_lead_bst'] = p2_bst
            d['exact_p2_lead_spe'] = p2_spe
            d['exact_p2_lead_bulk'] = p2_hp * (p2_def + p2_spd)
            d['exact_p2_lead_offense'] = max(p2_atk, p2_spa)
            d['exact_p2_lead_hp'] = p2_hp
            d['exact_p2_lead_atk'] = p2_atk
            d['exact_p2_lead_def'] = p2_def
            d['exact_p2_lead_spa'] = p2_spa
            d['exact_p2_lead_spd'] = p2_spd
            
            p1_bsts, p1_spes, p1_bulks, p1_offs = [], [], [], []
            p1_hps, p1_atks, p1_defs, p1_spas, p1_spds = [], [], [], [], []
            p1_type_advs, p1_type_disadvs = [], []
            
            for mon in p1_team:
                if not isinstance(mon, dict): continue
                m_name = str(mon.get('name', '')).lower()
                m_item = str(mon.get('item', 'none')).lower()
                m_types = [t.lower() for t in (mon.get('types') or []) if t and t.lower() != 'notype']
                
                m_hp = float(mon.get('base_hp', 80))
                m_atk = float(mon.get('base_atk', 80))
                m_def = float(mon.get('base_def', 80))
                m_spa = float(mon.get('base_spa', 80))
                m_spd = float(mon.get('base_spd', 80))
                m_spe = float(mon.get('base_spe', 80))
                m_bst = m_hp + m_atk + m_def + m_spa + m_spd + m_spe
                
                p1_names.append(m_name)
                p1_items.append(m_item)
                
                p1_bsts.append(m_bst); p1_spes.append(m_spe)
                p1_bulks.append(m_hp * (m_def + m_spd)); p1_offs.append(max(m_atk, m_spa))
                p1_hps.append(m_hp); p1_atks.append(m_atk); p1_defs.append(m_def); p1_spas.append(m_spa); p1_spds.append(m_spd)
                
                p1_type_advs.append(best_dual_type_mult(m_types, p2_lead_types))
                p1_type_disadvs.append(best_dual_type_mult(p2_lead_types, m_types))
                
            for stat_name, arr in [('bst', p1_bsts), ('spe', p1_spes), ('bulk', p1_bulks), ('off', p1_offs),
                                   ('hp', p1_hps), ('atk', p1_atks), ('def', p1_defs), ('spa', p1_spas), ('spd', p1_spds)]:
                d[f'exact_p1_{stat_name}_mean'] = np.mean(arr) if arr else 80.0
                d[f'exact_p1_{stat_name}_max'] = np.max(arr) if arr else 80.0
                d[f'exact_p1_{stat_name}_min'] = np.min(arr) if arr else 80.0
                d[f'exact_p1_{stat_name}_std'] = np.std(arr) if arr else 0.0
                
            d['exact_p1_leftovers_cnt'] = sum(1 for it in p1_items if it == 'leftovers')
            d['exact_p1_faster_than_lead_cnt'] = sum(1 for s in p1_spes if s > p2_spe)
            d['exact_p1_super_eff_vs_lead_cnt'] = sum(1 for a in p1_type_advs if a >= 2.0)
            d['exact_p1_resists_lead_cnt'] = sum(1 for dv in p1_type_disadvs if dv <= 0.5)
            d['exact_p1_max_adv_vs_lead'] = np.max(p1_type_advs) if p1_type_advs else 1.0
            d['exact_p1_mean_adv_vs_lead'] = np.mean(p1_type_advs) if p1_type_advs else 1.0
            
            # Turn-by-Turn Tracking
            p1_mon_last_hp = {m: 1.0 for m in p1_names}
            p2_mon_last_hp = {p2_lead_name: 1.0} if p2_lead_name != 'unknown' else {}
            p1_mon_last_status = {m: 'nostatus' for m in p1_names}
            p2_mon_last_status = {p2_lead_name: 'nostatus'} if p2_lead_name != 'unknown' else {}
            
            p1_seen_mons, p2_seen_mons = set(), set()
            p1_fainted_mons, p2_fainted_mons = set(), set()
            
            p1_switches, p2_switches = 0, 0
            p1_forced_switches, p2_forced_switches = 0, 0
            p1_no_move_turns, p2_no_move_turns = 0, 0
            
            p1_total_bp, p2_total_bp = 0.0, 0.0
            p1_eff_bp, p2_eff_bp = 0.0, 0.0
            p1_stat_scaled_dmg, p2_stat_scaled_dmg = 0.0, 0.0
            p1_phys_cnt, p1_spec_cnt, p1_stat_cnt = 0, 0, 0
            p2_phys_cnt, p2_spec_cnt, p2_stat_cnt = 0, 0, 0
            
            p1_recov_cnt, p2_recov_cnt = 0, 0
            p1_hazard_cnt, p2_hazard_cnt = 0, 0
            p1_setup_cnt, p2_setup_cnt = 0, 0
            p1_prio_cnt, p2_prio_cnt = 0, 0
            
            p1_dmg_dealt, p2_dmg_dealt = 0.0, 0.0
            p1_healed, p2_healed = 0.0, 0.0
            p1_se_hits, p2_se_hits = 0, 0
            p1_resisted_hits, p2_resisted_hits = 0, 0
            p1_immune_hits, p2_immune_hits = 0, 0
            p1_stab_moves, p2_stab_moves = 0, 0
            p1_turn_wins, p2_turn_wins = 0, 0
            
            p1_boost_sums, p2_boost_sums = [], []
            p1_active_hp_trace, p2_active_hp_trace = [], []
            p1_team_hp_trace, p2_team_hp_trace = [], []
            ko_diff_trace, seen_diff_trace = [], []
            
            p1_moves_seq, p2_moves_seq = [], []
            seq_tokens = []
            
            prev_n1, prev_n2 = None, None
            prev_hp1, prev_hp2 = 1.0, 1.0
            last_p1_mon = p1_names[0] if p1_names else 'unknown'
            last_p2_mon = p2_lead_name
            last_p1_move, last_p2_move = 'nomove', 'nomove'
            last_st1, last_st2 = 'nostatus', 'nostatus'
            
            p1_last_boost_sum, p2_last_boost_sum = 0.0, 0.0
            p1_last_atk_boost, p2_last_atk_boost = 0.0, 0.0
            p1_last_def_boost, p2_last_def_boost = 0.0, 0.0
            p1_last_spe_boost, p2_last_spe_boost = 0.0, 0.0
            p1_last_effects_cnt, p2_last_effects_cnt = 0, 0
            last_b1, last_b2 = {}, {}
            first_ko_advantage = 0.0
            
            for turn in timeline:
                if not isinstance(turn, dict): continue
                s1 = turn.get('p1_pokemon_state') or {}
                s2 = turn.get('p2_pokemon_state') or {}
                m1 = turn.get('p1_move_details')
                m2 = turn.get('p2_move_details')
                
                n1 = str(s1.get('name', '')).lower()
                n2 = str(s2.get('name', '')).lower()
                hp1 = float(s1.get('hp_pct', 1.0) if s1.get('hp_pct') is not None else 1.0)
                hp2 = float(s2.get('hp_pct', 1.0) if s2.get('hp_pct') is not None else 1.0)
                st1 = str(s1.get('status', 'nostatus')).lower()
                st2 = str(s2.get('status', 'nostatus')).lower()
                last_st1, last_st2 = st1, st2
                
                info1 = INTERNAL_POKEDEX.get(n1, DEFAULT_MON_INFO)
                info2 = INTERNAL_POKEDEX.get(n2, DEFAULT_MON_INFO)
                
                if prev_n1 is not None:
                    if n1 == prev_n1:
                        delta1 = hp1 - prev_hp1
                        if delta1 < 0: p2_dmg_dealt += (-delta1)
                        elif delta1 > 0: p1_healed += delta1
                    else:
                        p1_switches += 1
                        if prev_hp1 > 0.0: p1_forced_switches += 1
                        
                if prev_n2 is not None:
                    if n2 == prev_n2:
                        delta2 = hp2 - prev_hp2
                        if delta2 < 0: p1_dmg_dealt += (-delta2)
                        elif delta2 > 0: p2_healed += delta2
                    else:
                        p2_switches += 1
                        if prev_hp2 > 0.0: p2_forced_switches += 1
                        
                if n1:
                    p1_seen_mons.add(n1)
                    p1_mon_last_hp[n1] = hp1
                    p1_mon_last_status[n1] = st1
                    if hp1 <= 0.0 or st1 in ('fnt', 'fainted'):
                        p1_fainted_mons.add(n1)
                        p1_mon_last_hp[n1] = 0.0
                    last_p1_mon = n1
                    
                if n2:
                    p2_seen_mons.add(n2)
                    p2_mon_last_hp[n2] = hp2
                    p2_mon_last_status[n2] = st2
                    if hp2 <= 0.0 or st2 in ('fnt', 'fainted'):
                        p2_fainted_mons.add(n2)
                        p2_mon_last_hp[n2] = 0.0
                    last_p2_mon = n2
                    
                if first_ko_advantage == 0.0:
                    if len(p2_fainted_mons) > len(p1_fainted_mons): first_ko_advantage = 1.0
                    elif len(p1_fainted_mons) > len(p2_fainted_mons): first_ko_advantage = -1.0
                    
                prev_n1, prev_n2 = n1, n2
                prev_hp1, prev_hp2 = hp1, hp2
                
                p1_active_hp_trace.append(hp1)
                p2_active_hp_trace.append(hp2)
                
                p1_curr_total = sum(p1_mon_last_hp.values()) + max(0, 6 - len(p1_mon_last_hp)) * 1.0
                p2_curr_total = sum(p2_mon_last_hp.values()) + max(0, 6 - len(p2_mon_last_hp)) * 1.0
                p1_team_hp_trace.append(p1_curr_total)
                p2_team_hp_trace.append(p2_curr_total)
                ko_diff_trace.append(len(p2_fainted_mons) - len(p1_fainted_mons))
                seen_diff_trace.append(len(p2_seen_mons) - len(p1_seen_mons))
                
                if p1_curr_total > p2_curr_total: p1_turn_wins += 1
                elif p2_curr_total > p1_curr_total: p2_turn_wins += 1
                
                b1 = s1.get('boosts') or {}
                b2 = s2.get('boosts') or {}
                last_b1, last_b2 = b1, b2
                b1_sum = sum(float(v) for v in b1.values()) if isinstance(b1, dict) else 0.0
                b2_sum = sum(float(v) for v in b2.values()) if isinstance(b2, dict) else 0.0
                p1_boost_sums.append(b1_sum)
                p2_boost_sums.append(b2_sum)
                
                p1_last_boost_sum, p2_last_boost_sum = b1_sum, b2_sum
                p1_last_atk_boost = max(float(b1.get('atk', 0)), float(b1.get('spa', 0))) if isinstance(b1, dict) else 0.0
                p2_last_atk_boost = max(float(b2.get('atk', 0)), float(b2.get('spa', 0))) if isinstance(b2, dict) else 0.0
                p1_last_def_boost = max(float(b1.get('def', 0)), float(b1.get('spd', 0))) if isinstance(b1, dict) else 0.0
                p2_last_def_boost = max(float(b2.get('def', 0)), float(b2.get('spd', 0))) if isinstance(b2, dict) else 0.0
                p1_last_spe_boost = float(b1.get('spe', 0)) if isinstance(b1, dict) else 0.0
                p2_last_spe_boost = float(b2.get('spe', 0)) if isinstance(b2, dict) else 0.0
                
                eff1 = [e for e in (s1.get('effects') or []) if e and str(e).lower() != 'noeffect']
                eff2 = [e for e in (s2.get('effects') or []) if e and str(e).lower() != 'noeffect']
                p1_last_effects_cnt = len(eff1)
                p2_last_effects_cnt = len(eff2)
                
                if isinstance(m1, dict) and m1.get('name'):
                    mv1 = str(m1.get('name', '')).lower()
                    mt1 = str(m1.get('type', '')).lower()
                    mc1 = str(m1.get('category', '')).lower()
                    bp1 = float(m1.get('base_power', 0) or 0)
                    ac1 = float(m1.get('accuracy', 1.0) or 1.0)
                    pr1 = int(m1.get('priority', 0) or 0)
                    last_p1_move = mv1
                    
                    p1_moves_seq.append(mv1)
                    seq_tokens.append(f"p1m_{mv1}")
                    p1_total_bp += bp1
                    stab1 = 1.5 if mt1 in info1['types'] else 1.0
                    mult1 = get_type_mult(mt1, info2['types']) if bp1 > 0 else 1.0
                    eff_dmg1 = bp1 * ac1 * stab1 * mult1
                    p1_eff_bp += eff_dmg1
                    
                    if bp1 > 0:
                        if mc1 == 'physical':
                            stat_ratio1 = (info1['atk'] * STAGE_MULT.get(int(np.clip(b1.get('atk', 0), -6, 6)), 1.0)) / max(10.0, info2['def'] * STAGE_MULT.get(int(np.clip(b2.get('def', 0), -6, 6)), 1.0))
                            if st1 == 'brn': stat_ratio1 *= 0.5
                        else:
                            stat_ratio1 = (info1['spa'] * STAGE_MULT.get(int(np.clip(b1.get('spa', 0), -6, 6)), 1.0)) / max(10.0, info2['spd'] * STAGE_MULT.get(int(np.clip(b2.get('spd', 0), -6, 6)), 1.0))
                        p1_stat_scaled_dmg += eff_dmg1 * stat_ratio1
                        if mult1 >= 2.0: p1_se_hits += 1
                        elif mult1 == 0.0: p1_immune_hits += 1
                        elif mult1 <= 0.5: p1_resisted_hits += 1
                        if stab1 > 1.0: p1_stab_moves += 1
                        
                    if mc1 == 'physical': p1_phys_cnt += 1
                    elif mc1 == 'special': p1_spec_cnt += 1
                    else: p1_stat_cnt += 1
                    
                    if mv1 in RECOVERY_MOVES: p1_recov_cnt += 1
                    if mv1 in PHAZING_HAZARD_MOVES: p1_hazard_cnt += 1
                    if mv1 in SETUP_MOVES: p1_setup_cnt += 1
                    if pr1 > 0: p1_prio_cnt += 1
                else:
                    p1_no_move_turns += 1
                    last_p1_move = 'switch_or_idle'
                    
                if isinstance(m2, dict) and m2.get('name'):
                    mv2 = str(m2.get('name', '')).lower()
                    mt2 = str(m2.get('type', '')).lower()
                    mc2 = str(m2.get('category', '')).lower()
                    bp2 = float(m2.get('base_power', 0) or 0)
                    ac2 = float(m2.get('accuracy', 1.0) or 1.0)
                    pr2 = int(m2.get('priority', 0) or 0)
                    last_p2_move = mv2
                    
                    p2_moves_seq.append(mv2)
                    seq_tokens.append(f"p2m_{mv2}")
                    p2_total_bp += bp2
                    stab2 = 1.5 if mt2 in info2['types'] else 1.0
                    mult2 = get_type_mult(mt2, info1['types']) if bp2 > 0 else 1.0
                    eff_dmg2 = bp2 * ac2 * stab2 * mult2
                    p2_eff_bp += eff_dmg2
                    
                    if bp2 > 0:
                        if mc2 == 'physical':
                            stat_ratio2 = (info2['atk'] * STAGE_MULT.get(int(np.clip(b2.get('atk', 0), -6, 6)), 1.0)) / max(10.0, info1['def'] * STAGE_MULT.get(int(np.clip(b1.get('def', 0), -6, 6)), 1.0))
                            if st2 == 'brn': stat_ratio2 *= 0.5
                        else:
                            stat_ratio2 = (info2['spa'] * STAGE_MULT.get(int(np.clip(b2.get('spa', 0), -6, 6)), 1.0)) / max(10.0, info1['spd'] * STAGE_MULT.get(int(np.clip(b1.get('spd', 0), -6, 6)), 1.0))
                        p2_stat_scaled_dmg += eff_dmg2 * stat_ratio2
                        if mult2 >= 2.0: p2_se_hits += 1
                        elif mult2 == 0.0: p2_immune_hits += 1
                        elif mult2 <= 0.5: p2_resisted_hits += 1
                        if stab2 > 1.0: p2_stab_moves += 1
                        
                    if mc2 == 'physical': p2_phys_cnt += 1
                    elif mc2 == 'special': p2_spec_cnt += 1
                    else: p2_stat_cnt += 1
                    
                    if mv2 in RECOVERY_MOVES: p2_recov_cnt += 1
                    if mv2 in PHAZING_HAZARD_MOVES: p2_hazard_cnt += 1
                    if mv2 in SETUP_MOVES: p2_setup_cnt += 1
                    if pr2 > 0: p2_prio_cnt += 1
                else:
                    p2_no_move_turns += 1
                    last_p2_move = 'switch_or_idle'
                    
                seq_tokens.append(f"p1s_{n1}_{st1}")
                seq_tokens.append(f"p2s_{n2}_{st2}")

            # Aggregate Numerical + Checkpoint Features
            n_turns = max(1, len(timeline))
            d['exact_n_turns'] = len(timeline)
            d['exact_is_short_battle'] = 1.0 if len(timeline) < 30 else 0.0
            d['pkx_first_ko_adv'] = first_ko_advantage
            
            for cp in [4, 9, 14, 19, 24, 27, 28, 29]:
                if len(p1_team_hp_trace) > cp:
                    d[f'cp_hp_diff_t{cp+1}'] = p1_team_hp_trace[cp] - p2_team_hp_trace[cp]
                    d[f'cp_ko_diff_t{cp+1}'] = ko_diff_trace[cp]
                    d[f'cp_seen_diff_t{cp+1}'] = seen_diff_trace[cp]
                elif len(p1_team_hp_trace) > 0:
                    d[f'cp_hp_diff_t{cp+1}'] = p1_team_hp_trace[-1] - p2_team_hp_trace[-1]
                    d[f'cp_ko_diff_t{cp+1}'] = ko_diff_trace[-1]
                    d[f'cp_seen_diff_t{cp+1}'] = seen_diff_trace[-1]
                else:
                    d[f'cp_hp_diff_t{cp+1}'] = 0.0
                    d[f'cp_ko_diff_t{cp+1}'] = 0.0
                    d[f'cp_seen_diff_t{cp+1}'] = 0.0
                    
            d['cp_hp_accel_20_to_30'] = d['cp_hp_diff_t30'] - d['cp_hp_diff_t20']
            d['cp_hp_accel_25_to_30'] = d['cp_hp_diff_t30'] - d['cp_hp_diff_t25']
            d['cp_ko_accel_20_to_30'] = d['cp_ko_diff_t30'] - d['cp_ko_diff_t20']
            
            p1_total_hp_rem = sum(p1_mon_last_hp.values()) + max(0, 6 - len(p1_mon_last_hp)) * 1.0
            p2_total_hp_rem = sum(p2_mon_last_hp.values()) + max(0, 6 - len(p2_mon_last_hp)) * 1.0
            p1_alive_cnt = 6 - len(p1_fainted_mons)
            p2_alive_cnt = 6 - len(p2_fainted_mons)
            p1_healthy_cnt = sum(1 for v in p1_mon_last_hp.values() if v >= 0.5) + max(0, 6 - len(p1_mon_last_hp))
            p2_healthy_cnt = sum(1 for v in p2_mon_last_hp.values() if v >= 0.5) + max(0, 6 - len(p2_mon_last_hp))
            
            d['exact_p1_total_hp_rem'] = p1_total_hp_rem
            d['exact_p2_total_hp_rem'] = p2_total_hp_rem
            d['exact_total_hp_diff'] = p1_total_hp_rem - p2_total_hp_rem
            d['exact_total_hp_ratio'] = (p1_total_hp_rem + 0.1) / (p2_total_hp_rem + 0.1)
            d['exact_p1_fainted_cnt'] = len(p1_fainted_mons)
            d['exact_p2_fainted_cnt'] = len(p2_fainted_mons)
            d['exact_ko_diff'] = len(p2_fainted_mons) - len(p1_fainted_mons)
            d['exact_alive_ratio'] = (p1_alive_cnt + 0.1) / (p2_alive_cnt + 0.1)
            d['exact_healthy_diff'] = p1_healthy_cnt - p2_healthy_cnt
            d['exact_p1_seen_cnt'] = len(p1_seen_mons)
            d['exact_p2_seen_cnt'] = len(p2_seen_mons)
            d['exact_unrevealed_diff'] = (6 - len(p1_seen_mons)) - (6 - len(p2_seen_mons))
            
            p1_status_alive = sum(1 for m, st in p1_mon_last_status.items() if st != 'nostatus' and m not in p1_fainted_mons)
            p2_status_alive = sum(1 for m, st in p2_mon_last_status.items() if st != 'nostatus' and m not in p2_fainted_mons)
            p1_slp_frz = sum(1 for m, st in p1_mon_last_status.items() if st in ('slp', 'frz') and m not in p1_fainted_mons)
            p2_slp_frz = sum(1 for m, st in p2_mon_last_status.items() if st in ('slp', 'frz') and m not in p2_fainted_mons)
            
            d['exact_p1_status_alive'] = p1_status_alive
            d['exact_p2_status_alive'] = p2_status_alive
            d['exact_status_diff'] = p2_status_alive - p1_status_alive
            d['exact_slp_frz_diff'] = p2_slp_frz - p1_slp_frz
            
            p1_sev = sum(STATUS_SEVERITY.get(st, 0.0) for m, st in p1_mon_last_status.items() if m not in p1_fainted_mons)
            p2_sev = sum(STATUS_SEVERITY.get(st, 0.0) for m, st in p2_mon_last_status.items() if m not in p2_fainted_mons)
            d['pkx_status_severity_diff'] = p2_sev - p1_sev
            
            # Decoded P1 vs P2 Remaining Roster Strength
            p1_rem_bst, p1_rem_bulk, p1_rem_off = 0.0, 0.0, 0.0
            for m, h_val in p1_mon_last_hp.items():
                if h_val > 0.0:
                    inf = INTERNAL_POKEDEX.get(m, DEFAULT_MON_INFO)
                    st_pen = 0.6 if p1_mon_last_status.get(m, 'nostatus') in ('slp', 'frz', 'brn', 'par') else 1.0
                    p1_rem_bst += h_val * inf['bst'] * st_pen
                    p1_rem_bulk += h_val * inf['bulk']
                    p1_rem_off += h_val * inf['off'] * st_pen
                    
            p2_rem_bst, p2_rem_bulk, p2_rem_off = 0.0, 0.0, 0.0
            p2_revealed_bsts = []
            for m, h_val in p2_mon_last_hp.items():
                inf = INTERNAL_POKEDEX.get(m, DEFAULT_MON_INFO)
                p2_revealed_bsts.append(inf['bst'])
                if h_val > 0.0:
                    st_pen = 0.6 if p2_mon_last_status.get(m, 'nostatus') in ('slp', 'frz', 'brn', 'par') else 1.0
                    p2_rem_bst += h_val * inf['bst'] * st_pen
                    p2_rem_bulk += h_val * inf['bulk']
                    p2_rem_off += h_val * inf['off'] * st_pen
                    
            p2_unrev_cnt = max(0, 6 - len(p2_mon_last_hp))
            p2_rem_bst += p2_unrev_cnt * 515.0
            p2_rem_bulk += p2_unrev_cnt * 18500.0
            p2_rem_off += p2_unrev_cnt * 110.0
            
            d['pkx_p1_rem_bst'] = p1_rem_bst
            d['pkx_p2_rem_bst'] = p2_rem_bst
            d['pkx_rem_bst_diff'] = p1_rem_bst - p2_rem_bst
            d['pkx_rem_bst_ratio'] = (p1_rem_bst + 10.0) / (p2_rem_bst + 10.0)
            d['pkx_rem_bulk_diff'] = p1_rem_bulk - p2_rem_bulk
            d['pkx_rem_off_diff'] = p1_rem_off - p2_rem_off
            d['pkx_p2_revealed_bst_mean'] = np.mean(p2_revealed_bsts) if p2_revealed_bsts else 515.0
            
            act1 = INTERNAL_POKEDEX.get(last_p1_mon, DEFAULT_MON_INFO)
            act2 = INTERNAL_POKEDEX.get(last_p2_mon, DEFAULT_MON_INFO)
            spe1_mult = STAGE_MULT.get(int(np.clip(last_b1.get('spe', 0), -6, 6)), 1.0) * (0.25 if last_st1 == 'par' else 1.0)
            spe2_mult = STAGE_MULT.get(int(np.clip(last_b2.get('spe', 0), -6, 6)), 1.0) * (0.25 if last_st2 == 'par' else 1.0)
            
            outspeeds = 1 if (act1['spe'] * spe1_mult) > (act2['spe'] * spe2_mult) else -1
            type_adv_1v2 = best_dual_type_mult(act1['types'], act2['types'])
            type_adv_2v1 = best_dual_type_mult(act2['types'], act1['types'])
            
            d['pkx_last_active_outspeeds'] = float(outspeeds)
            d['pkx_last_active_type_adv'] = type_adv_1v2 - type_adv_2v1
            d['pkx_last_active_bst_diff'] = act1['bst'] - act2['bst']
            
            d['exact_last_active_hp_diff'] = (p1_active_hp_trace[-1] - p2_active_hp_trace[-1]) if p1_active_hp_trace else 0.0
            d['exact_last_boost_diff'] = p1_last_boost_sum - p2_last_boost_sum
            d['exact_last_atk_boost_diff'] = p1_last_atk_boost - p2_last_atk_boost
            d['exact_last_def_boost_diff'] = p1_last_def_boost - p2_last_def_boost
            d['exact_last_spe_boost_diff'] = p1_last_spe_boost - p2_last_spe_boost
            d['exact_mean_boost_diff'] = (np.mean(p1_boost_sums) - np.mean(p2_boost_sums)) if p1_boost_sums else 0.0
            d['exact_last_effects_diff'] = p1_last_effects_cnt - p2_last_effects_cnt
            
            if len(p1_team_hp_trace) >= 3:
                t_third = max(1, len(p1_team_hp_trace) // 3)
                early_diff = np.mean(p1_team_hp_trace[:t_third]) - np.mean(p2_team_hp_trace[:t_third])
                mid_diff = np.mean(p1_team_hp_trace[t_third:2*t_third]) - np.mean(p2_team_hp_trace[t_third:2*t_third])
                late_diff = np.mean(p1_team_hp_trace[-t_third:]) - np.mean(p2_team_hp_trace[-t_third:])
                diffs = np.array(p1_team_hp_trace) - np.array(p2_team_hp_trace)
                weights = np.exp(np.linspace(-2.0, 0.0, len(diffs)))
                weights /= np.sum(weights)
                
                d['exact_team_hp_diff_early'] = early_diff
                d['exact_team_hp_diff_mid'] = mid_diff
                d['exact_team_hp_diff_late'] = late_diff
                d['exact_team_hp_velocity_late_early'] = late_diff - early_diff
                d['exact_team_hp_velocity_late_mid'] = late_diff - mid_diff
                d['exact_team_hp_trend_slope'] = np.polyfit(np.arange(len(diffs)), diffs, 1)[0] if len(diffs) > 1 else 0.0
                d['pkx_exp_weighted_hp_diff'] = float(np.sum(diffs * weights))
                d['pkx_hp_diff_std'] = float(np.std(diffs))
            else:
                for k_zero in ['exact_team_hp_diff_early', 'exact_team_hp_diff_mid', 'exact_team_hp_diff_late',
                               'exact_team_hp_velocity_late_early', 'exact_team_hp_velocity_late_mid',
                               'exact_team_hp_trend_slope', 'pkx_exp_weighted_hp_diff', 'pkx_hp_diff_std']:
                    d[k_zero] = 0.0
                    
            p1_slots = sorted(list(p1_mon_last_hp.values()) + [1.0] * max(0, 6 - len(p1_mon_last_hp)), reverse=True)[:6]
            p2_slots = sorted(list(p2_mon_last_hp.values()) + [1.0] * max(0, 6 - len(p2_mon_last_hp)), reverse=True)[:6]
            for s_i in range(6):
                d[f'pkx_slot_{s_i}_hp_diff'] = p1_slots[s_i] - p2_slots[s_i]
                
            d['exact_switch_diff'] = p1_switches - p2_switches
            d['pkx_forced_switch_diff'] = p2_forced_switches - p1_forced_switches
            d['exact_no_move_diff'] = p2_no_move_turns - p1_no_move_turns
            d['exact_total_bp_diff'] = p1_total_bp - p2_total_bp
            d['exact_eff_bp_diff'] = p1_eff_bp - p2_eff_bp
            d['pkx_stat_scaled_dmg_diff'] = p1_stat_scaled_dmg - p2_stat_scaled_dmg
            d['exact_bp_per_turn_diff'] = (p1_total_bp - p2_total_bp) / n_turns
            d['exact_recov_diff'] = p1_recov_cnt - p2_recov_cnt
            d['exact_hazard_diff'] = p1_hazard_cnt - p2_hazard_cnt
            d['exact_setup_diff'] = p1_setup_cnt - p2_setup_cnt
            d['exact_prio_diff'] = p1_prio_cnt - p2_prio_cnt
            d['exact_off_move_ratio_diff'] = ((p1_phys_cnt + p1_spec_cnt) - (p2_phys_cnt + p2_spec_cnt)) / n_turns
            d['pkx_realized_dmg_diff'] = p1_dmg_dealt - p2_dmg_dealt
            d['pkx_realized_heal_diff'] = p1_healed - p2_healed
            d['pkx_net_attrition_diff'] = (p1_dmg_dealt + p1_healed) - (p2_dmg_dealt + p2_healed)
            d['pkx_se_hits_diff'] = p1_se_hits - p2_se_hits
            d['pkx_resisted_hits_diff'] = p2_resisted_hits - p1_resisted_hits
            d['pkx_immune_hits_diff'] = p2_immune_hits - p1_immune_hits
            d['pkx_stab_moves_diff'] = p1_stab_moves - p2_stab_moves
            d['pkx_turn_lead_ratio'] = (p1_turn_wins - p2_turn_wins) / n_turns
            
            p1_best_alive = max([m for m in p1_names if m not in p1_fainted_mons] or ['none'],
                                key=lambda x: p1_mon_last_hp.get(x, 0.0) * INTERNAL_POKEDEX.get(x, DEFAULT_MON_INFO)['bst'])
            p2_best_alive = max([m for m in p2_seen_mons if m not in p2_fainted_mons] or ['none'],
                                key=lambda x: p2_mon_last_hp.get(x, 0.0) * INTERNAL_POKEDEX.get(x, DEFAULT_MON_INFO)['bst'])
            
            hp_bucket = int(np.clip(np.round(p1_total_hp_rem - p2_total_hp_rem), -5, 5))
            active_hp_b1 = int(np.round((p1_active_hp_trace[-1] if p1_active_hp_trace else 1.0) * 4))
            active_hp_b2 = int(np.round((p2_active_hp_trace[-1] if p2_active_hp_trace else 1.0) * 4))
            
            rows_feats.append(d)
            cat_rows.append({
                'cat_p1_lead': p1_names[0] if p1_names else 'none',
                'cat_p2_lead': p2_lead_name,
                'cat_lead_pair': f"{p1_names[0] if p1_names else 'none'}_vs_{p2_lead_name}",
                'cat_last_p1': last_p1_mon,
                'cat_last_p2': last_p2_mon,
                'cat_last_pair': f"{last_p1_mon}_vs_{last_p2_mon}",
                'cat_last_status_pair': f"{last_st1}_vs_{last_st2}",
                'cat_last_move_pair': f"{last_p1_move}_vs_{last_p2_move}",
                'cat_best_alive_pair': f"{p1_best_alive}_vs_{p2_best_alive}",
                'cat_ko_state': f"{len(p1_fainted_mons)}_vs_{len(p2_fainted_mons)}",
                'cat_seen_state': f"{len(p1_seen_mons)}_vs_{len(p2_seen_mons)}",
                'cat_status_cnt_state': f"{p1_status_alive}_vs_{p2_status_alive}",
                'cat_hp_bucket_ko': f"hp_{hp_bucket}_ko_{len(p2_fainted_mons)-len(p1_fainted_mons)}",
                'cat_last_p1_st': f"{last_p1_mon}_{last_st1}",
                'cat_last_p2_st': f"{last_p2_mon}_{last_st2}",
                'cat_active_hp_tier_pair': f"{active_hp_b1}_vs_{active_hp_b2}",
                'cat_p1_best_alive': p1_best_alive,
                'cat_p2_best_alive': p2_best_alive,
                'cat_active_matchup_mechanic': f"spd_{outspeeds}_adv_{int(type_adv_1v2 >= 2.0)}_dis_{int(type_adv_2v1 >= 2.0)}",
                'cat_first_ko_state': f"fko_{int(first_ko_advantage)}_ko_{len(p1_fainted_mons)}_{len(p2_fainted_mons)}"
            })
            token_records.append({
                'p1_mons': list(set(p1_names)),
                'p1_alive_mons': [m for m in p1_names if m not in p1_fainted_mons],
                'p1_fainted': list(p1_fainted_mons),
                'p2_seen': list(p2_seen_mons),
                'p2_alive_seen': [m for m in p2_seen_mons if m not in p2_fainted_mons],
                'p2_fainted_mons': list(p2_fainted_mons),
                'p2_lead': p2_lead_name,
                'lead_matchup': f"{p1_names[0] if p1_names else 'none'}_vs_{p2_lead_name}",
                'last_matchup': f"{last_p1_mon}_vs_{last_p2_mon}",
                'best_alive_pair': f"{p1_best_alive}_vs_{p2_best_alive}",
                'p1_moves': list(set(p1_moves_seq)),
                'p2_moves': list(set(p2_moves_seq)),
                'all_matchups': [f"{m1}_vs_{m2}" for m1 in p1_names for m2 in p2_seen_mons]
            })
            corpus_texts.append(" ".join(seq_tokens + [f"p1r_{m}" for m in p1_names] + [f"p2r_{m}" for m in p2_seen_mons]))
            
    return pd.DataFrame(rows_feats), pd.DataFrame(cat_rows), token_records, corpus_texts, np.array(targets)

uni_train_df, cat_train_df, train_toks, tr_texts, y = extract_v4_features(TRAIN_PATH, is_train=True)
uni_test_df, cat_test_df, test_toks, te_texts, _ = extract_v4_features(TEST_PATH, is_train=False)

# 4. 10-Fold Out-of-Fold Target Encoding + TF-IDF Probes + SVD
print("🎯 Computing 10-Fold Target Encodings & Battle-Log Text Probes...")
N_FOLDS = 10
skf_10 = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
prior = np.mean(y)

tfidf = TfidfVectorizer(max_features=3500, ngram_range=(1, 2), sublinear_tf=True)
tr_tfidf = tfidf.fit_transform(tr_texts)
te_tfidf = tfidf.transform(te_texts)

svd = TruncatedSVD(n_components=20, random_state=SEED)
tr_svd = pd.DataFrame(svd.fit_transform(tr_tfidf), columns=[f"pkx_svd_{i}" for i in range(20)])
te_svd = pd.DataFrame(svd.transform(te_tfidf), columns=[f"pkx_svd_{i}" for i in range(20)])

te_keys = [
    ('p1_mons', 25.0), ('p1_alive_mons', 20.0), ('p1_fainted', 20.0),
    ('p2_seen', 25.0), ('p2_alive_seen', 20.0), ('p2_fainted_mons', 20.0),
    ('p2_lead', 25.0), ('lead_matchup', 18.0), ('last_matchup', 18.0),
    ('best_alive_pair', 18.0), ('p1_moves', 25.0), ('p2_moves', 25.0), ('all_matchups', 30.0)
]

te_tr_mat = np.zeros((len(y), len(te_keys) * 2 + 2), dtype=np.float64)
te_te_mat = np.zeros((len(te_texts), len(te_keys) * 2 + 2), dtype=np.float64)

for fold, (tr_idx, va_idx) in enumerate(skf_10.split(uni_train_df, y)):
    tr_sub = [train_toks[i] for i in tr_idx]
    va_sub = [train_toks[i] for i in va_idx]
    y_tr = y[tr_idx]
    
    for k_i, (k_name, sm) in enumerate(te_keys):
        counts = defaultdict(lambda: [0.0, 0.0])
        for rec, targ in zip(tr_sub, y_tr):
            val = rec[k_name]
            items = val if isinstance(val, list) else [val]
            for it in items:
                if not it: continue
                counts[it][0] += targ
                counts[it][1] += 1.0
        rmap = {k: (s + sm * prior) / (c + sm) for k, (s, c) in counts.items()}
        
        for row_j, rec in enumerate(va_sub):
            val = rec[k_name]
            items = val if isinstance(val, list) else [val]
            sc = [rmap.get(it, prior) for it in items if it] or [prior]
            te_tr_mat[va_idx[row_j], k_i * 2] = np.mean(sc)
            te_tr_mat[va_idx[row_j], k_i * 2 + 1] = np.max(sc)
            
        for row_j, rec in enumerate(test_toks):
            val = rec[k_name]
            items = val if isinstance(val, list) else [val]
            sc = [rmap.get(it, prior) for it in items if it] or [prior]
            te_te_mat[row_j, k_i * 2] += np.mean(sc) / N_FOLDS
            te_te_mat[row_j, k_i * 2 + 1] += np.max(sc) / N_FOLDS
            
    clf_txt1 = LogisticRegression(C=0.20, max_iter=500)
    clf_txt1.fit(tr_tfidf[tr_idx], y_tr)
    te_tr_mat[va_idx, -2] = clf_txt1.predict_proba(tr_tfidf[va_idx])[:, 1]
    te_te_mat[:, -2] += clf_txt1.predict_proba(te_tfidf)[:, 1] / N_FOLDS

    clf_txt2 = RidgeClassifier(alpha=3.0, random_state=SEED)
    clf_txt2.fit(tr_tfidf[tr_idx], y_tr)
    te_tr_mat[va_idx, -1] = clf_txt2.decision_function(tr_tfidf[va_idx])
    te_te_mat[:, -1] += clf_txt2.decision_function(te_tfidf) / N_FOLDS

te_col_names = []
for k_name, _ in te_keys:
    te_col_names.extend([f"exact_te_{k_name}_mean", f"exact_te_{k_name}_max"])
te_col_names.extend(["pkx_oof_text_logreg", "pkx_oof_text_ridge"])

te_tr_df = pd.DataFrame(te_tr_mat, columns=te_col_names)
te_te_df = pd.DataFrame(te_te_mat, columns=te_col_names)
te_tr_df['exact_te_alive_vs_p2_seen'] = te_tr_df['exact_te_p1_alive_mons_mean'] - (1.0 - te_tr_df['exact_te_p2_seen_mean'])
te_te_df['exact_te_alive_vs_p2_seen'] = te_te_df['exact_te_p1_alive_mons_mean'] - (1.0 - te_te_df['exact_te_p2_seen_mean'])
te_tr_df['exact_te_move_quality_diff'] = te_tr_df['exact_te_p1_moves_mean'] - (1.0 - te_tr_df['exact_te_p2_moves_mean'])
te_te_df['exact_te_move_quality_diff'] = te_te_df['exact_te_p1_moves_mean'] - (1.0 - te_te_df['exact_te_p2_moves_mean'])

# 5. Merge & Prune Features Strictly on Training Set
if 'X' in globals() and 'X_test' in globals() and isinstance(globals()['X'], pd.DataFrame) and len(globals()['X']) == len(y):
    X_full = pd.concat([globals()['X'].reset_index(drop=True), uni_train_df, te_tr_df, tr_svd], axis=1)
    X_test_full = pd.concat([globals()['X_test'].reset_index(drop=True), uni_test_df, te_te_df, te_svd], axis=1)
else:
    X_full = pd.concat([uni_train_df, te_tr_df, tr_svd], axis=1)
    X_test_full = pd.concat([uni_test_df, te_te_df, te_svd], axis=1)

X_full = X_full.loc[:, ~X_full.columns.duplicated()]
X_test_full = X_test_full.loc[:, ~X_test_full.columns.duplicated()]
X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test_full = X_test_full.replace([np.inf, -np.inf], np.nan).fillna(0)

non_const = X_full.columns[X_full.nunique() > 1]
X_full = X_full[non_const]
X_test_full = X_test_full[non_const]

if X_full.shape[1] > 320:
    sel_lgb = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.6, random_state=SEED, n_jobs=-1, verbose=-1)
    sel_lgb.fit(X_full, y)
    imp = pd.Series(sel_lgb.feature_importances_, index=X_full.columns)
    domain_cols = [c for c in X_full.columns if c.startswith('exact_') or c.startswith('pkx_') or c.startswith('cp_')]
    top_cols = list(dict.fromkeys(domain_cols + imp.nlargest(450).index.tolist()))
    X_num = X_full[top_cols].copy()
    X_test_num = X_test_full[top_cols].copy()
else:
    X_num = X_full.copy()
    X_test_num = X_test_full.copy()

cat_col_names = list(cat_train_df.columns)
X_cb_train = pd.concat([X_num, cat_train_df.astype(str)], axis=1)
X_cb_test = pd.concat([X_test_num, cat_test_df.astype(str)], axis=1)

print(f"✅ V4 Pure CatBoost Specialist Matrix: {X_num.shape[1]} numerical + {len(cat_col_names)} tactical categoricals!")

# 6. Train 10-Fold Pure Shallow-to-Medium CatBoost Specialist Ensemble (Zero Weak LGBM/XGB Dilution!)
oof_preds = np.zeros((len(X_num), 4), dtype=np.float64)
test_preds = np.zeros((len(X_test_num), 4), dtype=np.float64)

print(f"\n🔄 Training 10-Fold Pure CatBoost Specialist Ensemble (D4-Ordered, D5-Ordered-A, D5-Ordered-B, D6-Ordered)...", flush=True)
for fold, (tr_idx, va_idx) in enumerate(skf_10.split(X_num, y), 1):
    f_t0 = time.time()
    Xcb_tr, Xcb_va = X_cb_train.iloc[tr_idx], X_cb_train.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    # Specialist 0: CatBoost Ordered Depth 4 (High Generalization, Zero Overfitting)
    cb_d4 = CatBoostClassifier(
        iterations=5000, learning_rate=0.024, depth=4, l2_leaf_reg=5,
        loss_function="Logloss", eval_metric="AUC", cat_features=cat_col_names,
        random_seed=SEED + fold, allow_writing_files=False, thread_count=-1, verbose=False
    )
    cb_d4.fit(Xcb_tr, y_tr, eval_set=(Xcb_va, y_va), early_stopping_rounds=250, verbose=False)
    oof_preds[va_idx, 0] = cb_d4.predict_proba(Xcb_va)[:, 1]
    test_preds[:, 0] += cb_d4.predict_proba(X_cb_test)[:, 1] / N_FOLDS

    # Specialist 1: CatBoost Ordered Depth 5 - Seed A (Your 0.72470 Champion Architecture!)
    cb_d5_a = CatBoostClassifier(
        iterations=4500, learning_rate=0.022, depth=5, l2_leaf_reg=6,
        loss_function="Logloss", eval_metric="AUC", cat_features=cat_col_names,
        random_seed=SEED + 100 + fold, allow_writing_files=False, thread_count=-1, verbose=False
    )
    cb_d5_a.fit(Xcb_tr, y_tr, eval_set=(Xcb_va, y_va), early_stopping_rounds=250, verbose=False)
    oof_preds[va_idx, 1] = cb_d5_a.predict_proba(Xcb_va)[:, 1]
    test_preds[:, 1] += cb_d5_a.predict_proba(X_cb_test)[:, 1] / N_FOLDS

    # Specialist 2: CatBoost Ordered Depth 5 - Seed B (Higher L2 Regularization & Feature Randomness)
    cb_d5_b = CatBoostClassifier(
        iterations=4500, learning_rate=0.020, depth=5, l2_leaf_reg=10, random_strength=1.5,
        loss_function="Logloss", eval_metric="AUC", cat_features=cat_col_names,
        random_seed=SEED + 200 + fold, allow_writing_files=False, thread_count=-1, verbose=False
    )
    cb_d5_b.fit(Xcb_tr, y_tr, eval_set=(Xcb_va, y_va), early_stopping_rounds=250, verbose=False)
    oof_preds[va_idx, 2] = cb_d5_b.predict_proba(Xcb_va)[:, 1]
    test_preds[:, 2] += cb_d5_b.predict_proba(X_cb_test)[:, 1] / N_FOLDS

    # Specialist 3: CatBoost Ordered Depth 6
    cb_d6 = CatBoostClassifier(
        iterations=4000, learning_rate=0.018, depth=6, l2_leaf_reg=10,
        loss_function="Logloss", eval_metric="AUC", cat_features=cat_col_names,
        random_seed=SEED + 300 + fold, allow_writing_files=False, thread_count=-1, verbose=False
    )
    cb_d6.fit(Xcb_tr, y_tr, eval_set=(Xcb_va, y_va), early_stopping_rounds=250, verbose=False)
    oof_preds[va_idx, 3] = cb_d6.predict_proba(Xcb_va)[:, 1]
    test_preds[:, 3] += cb_d6.predict_proba(X_cb_test)[:, 1] / N_FOLDS

    f_blend = np.average(oof_preds[va_idx], axis=1, weights=[0.25, 0.30, 0.30, 0.15])
    print(f"✅ Fold {fold:2d}/{N_FOLDS} ({time.time()-f_t0:.0f}s) | Val Acc: {accuracy_score(y_va, (f_blend >= 0.5).astype(int)):.5f} | Val AUC: {roc_auc_score(y_va, f_blend):.5f}", flush=True)

np.save("/kaggle/working/oof_preds_v4_pure_cb.npy", oof_preds)
np.save("/kaggle/working/test_preds_v4_pure_cb.npy", test_preds)

print("\n================ 10-FOLD PURE CATBOOST BREAKDOWN ================")
m_names = ["CatBoost-D4", "CatBoost-D5-SeedA", "CatBoost-D5-SeedB", "CatBoost-D6"]
for idx, m_name in enumerate(m_names):
    print(f"   • {m_name:<18} | OOF Acc (@0.50): {accuracy_score(y, (oof_preds[:, idx] >= 0.5).astype(int)):.5f} | OOF AUC: {roc_auc_score(y, oof_preds[:, idx]):.5f}")

# 7. Log-Odds Weighted Blend of the 4 Pure CatBoost Specialists at Strict 0.50 Threshold
eps = 1e-6
oof_logits = np.log(np.clip(oof_preds, eps, 1 - eps) / (1.0 - np.clip(oof_preds, eps, 1 - eps)))
test_logits = np.log(np.clip(test_preds, eps, 1 - eps) / (1.0 - np.clip(test_preds, eps, 1 - eps)))

weights = np.array([0.25, 0.30, 0.30, 0.15])
final_oof_prob = 1.0 / (1.0 + np.exp(-np.dot(oof_logits, weights)))
final_test_prob = 1.0 / (1.0 + np.exp(-np.dot(test_logits, weights)))

oof_acc_050 = accuracy_score(y, (final_oof_prob >= 0.50).astype(int))
print("---------------------------------------------------------------")
print(f"🚀 Final 10-Fold Pure CatBoost OOF Acc (@0.500): {oof_acc_050:.5f}")
print(f"🏆 Final 10-Fold Pure CatBoost OOF ROC-AUC     : {roc_auc_score(y, final_oof_prob):.5f}")

final_preds = (final_test_prob >= 0.50).astype(int)
submission = pd.read_csv(f"{BASE}/sample_submission.csv")
submission['player_won'] = final_preds
submission.to_csv("/kaggle/working/submission.csv", index=False)

print(f"\n✅ Saved Pure CatBoost Specialist submission to /kaggle/working/submission.csv!")
print("Final Test Prediction Distribution:", Counter(final_preds))

📚 Building Internal Pokédex strictly from train.jsonl...
   ✅ Indexed 239 species from train.jsonl!
🚀 Extracting V2 Metagame + V3 Checkpoint Trajectories + 20 Tactical Categoricals...
🎯 Computing 10-Fold Target Encodings & Battle-Log Text Probes...
✅ V4 Pure CatBoost Specialist Matrix: 492 numerical + 20 tactical categoricals!

🔄 Training 10-Fold Pure CatBoost Specialist Ensemble (D4-Ordered, D5-Ordered-A, D5-Ordered-B, D6-Ordered)...
✅ Fold  1/10 (588s) | Val Acc: 0.72300 | Val AUC: 0.80346
✅ Fold  2/10 (805s) | Val Acc: 0.72000 | Val AUC: 0.79717
✅ Fold  3/10 (824s) | Val Acc: 0.70600 | Val AUC: 0.76904
✅ Fold  4/10 (430s) | Val Acc: 0.71800 | Val AUC: 0.80152
✅ Fold  5/10 (774s) | Val Acc: 0.73200 | Val AUC: 0.80798
✅ Fold  6/10 (729s) | Val Acc: 0.72200 | Val AUC: 0.78676
✅ Fold  7/10 (580s) | Val Acc: 0.73300 | Val AUC: 0.79955
✅ Fold  8/10 (708s) | Val Acc: 0.73200 | Val AUC: 0.81942
✅ Fold  9/10 (581s) | Val Acc: 0.70100 | Val AUC: 0.78611
✅ Fold 10/10 (635s) | Val Acc: 0.71100 